<a href="https://colab.research.google.com/github/netsetos/genai-engg-gcp-learners/blob/main/module-07-mcp-and-cloud-run/lesson-7.3-agent-mcp/notebooks/GCP_Capstone_7.3_AgentMCP.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 7.3 Connect Agent to Remote MCP — McpToolset, IAM Auth, BQ & Doc AI
**Netsetos GenAI Engineering — GCP Capstone**

Wire ADK agents to Cloud Run MCP servers. Multi-server architecture with Gemini routing.


## Setup

> **If `cannot import name McpToolset` persists after running this cell:** your runtime cached an older google-adk from a previous run. `pip` cannot reload an already-imported module, so do **Runtime -> Restart session**, then run Setup again. (A fresh runtime installs the pinned `google-adk>=1.15` before any import, so this only bites if you ran a cell before upgrading.)

In [ ]:
!pip install -q -U "google-adk>=1.15" "mcp>=1.24,<2" google-genai google-auth "fastmcp>=4,<5"

from google.colab import auth
auth.authenticate_user()

PROJECT_ID = 'documind-ai-YOUR-ID'  # CHANGE

import os
os.environ['GOOGLE_CLOUD_PROJECT'] = PROJECT_ID
os.environ['GOOGLE_CLOUD_LOCATION'] = 'global'  # Gemini 3.x generation is served from the global Vertex endpoint
os.environ['GOOGLE_GENAI_USE_VERTEXAI'] = 'TRUE'
print(f'Project: {PROJECT_ID}')


## Cell 1: Connect to Local MCP Server


In [ ]:
# First, start your Lesson 7.1 server locally or via proxy
# For local: python documind_server.py
# For proxy: gcloud run services proxy documind-mcp-server --port=3000

# Import from the mcp_toolset SUBMODULE (needs the mcp client SDK, installed in Setup).
try:
    from google.adk.tools.mcp_tool.mcp_toolset import McpToolset            # google-adk >= 1.15 (canonical)
except ImportError:                                                          # older ADK: all-caps class name
    from google.adk.tools.mcp_tool.mcp_toolset import MCPToolset as McpToolset
from google.adk.tools.mcp_tool.mcp_session_manager import StreamableHTTPConnectionParams

# Connect to local/proxy server (no auth needed)
local_tools = McpToolset(
    connection_params=StreamableHTTPConnectionParams(
        url='http://localhost:8000/mcp',  # or localhost:3000 for proxy
    ),
)
print('McpToolset created (connects when agent starts)')


## Cell 2: Create Agent with MCP Tools


In [ ]:
from google.adk.agents import LlmAgent

agent = LlmAgent(
    model='gemini-3.6-flash',
    name='documind_agent',
    instruction='You are DocuMind AI. Use tools to search documents, '
                'calculate costs, and classify content.',
    tools=[local_tools],
)
print(f'Agent created: {agent.name}')


## Cell 3: IAM Token for Cloud Run


In [ ]:
import google.auth.transport.requests
import google.oauth2.id_token

# Replace with your Cloud Run URL
CLOUD_RUN_URL = 'https://documind-mcp-server-HASH-uc.a.run.app'
MCP_URL = f'{CLOUD_RUN_URL}/mcp'

def get_id_token(url):
    audience = url.split('/mcp')[0]
    request = google.auth.transport.requests.Request()
    return google.oauth2.id_token.fetch_id_token(request, audience)

try:
    token = get_id_token(MCP_URL)
    print(f'Token generated: {token[:30]}...')
except Exception as e:
    print(f'Token error (expected if no Cloud Run deployed): {e}')
    token = 'dummy-for-local-testing'


## Cell 4: Connect to Authenticated Cloud Run MCP


In [ ]:
# Connect to Cloud Run with IAM authentication
auth_tools = McpToolset(
    connection_params=StreamableHTTPConnectionParams(
        url=MCP_URL,
        headers={'Authorization': f'Bearer {token}'},
    ),
)

agent_cloud = LlmAgent(
    model='gemini-3.6-flash',
    name='documind_cloud_agent',
    instruction='You are DocuMind AI connected to Cloud Run tools.',
    tools=[auth_tools],
)
print(f'Cloud agent created with authenticated MCP connection')


## Cell 5: BigQuery via MCP Toolbox


In [ ]:
# If you have Toolbox deployed from Lesson 7.2:
TOOLBOX_URL = 'https://documind-toolbox-HASH-uc.a.run.app'

try:
    _tok = get_id_token(TOOLBOX_URL)
except Exception:
    _tok = 'PLACEHOLDER-TOKEN'  # deploy the Lesson 7.2 service, set the URL above, and run: gcloud auth application-default login

# BQ Toolbox with tool filtering
bq_tools = McpToolset(
    connection_params=StreamableHTTPConnectionParams(
        url=f'{TOOLBOX_URL}/mcp',
        headers={'Authorization': f'Bearer {_tok}'},
    ),
    tool_filter=['query-document-analytics', 'predict-document-category'],
)
print('BigQuery MCP Toolset configured')

# BQML tools.yaml example for predict-document-category:
print('''
tools.yaml snippet for BQML prediction:
---
kind: tool
name: predict-document-category
type: bigquery-sql
source: documind-bq
description: Predict document category using BQML model.
parameters:
  - name: document_text
    type: string
statement: >
  SELECT * FROM ML.PREDICT(
    MODEL `documind.document_classifier`,
    (SELECT @document_text AS text_content));
''')



## Cell 6: Multi-Server Agent


In [ ]:
# Combine multiple MCP servers in one agent

def format_report(data: dict, template: str = 'default') -> str:
    """Format extracted data into a structured report."""
    return f'Report ({template}): {data}'

# Multi-MCP agent (use local URLs for testing)
multi_agent = LlmAgent(
    model='gemini-3.6-flash',
    name='documind_multi_agent',
    instruction="""You are DocuMind AI with access to:
- Document search, cost, and classification tools
- BigQuery analytics and ML prediction tools
- Report formatting
Use the right tool for each query.""",
    tools=[
        local_tools,        # FastMCP server
        # bq_tools,         # BigQuery Toolbox (uncomment when deployed)
        format_report,       # Direct function tool
    ],
)
print(f'Multi-server agent: {multi_agent.name}')
print(f'Tools: {len(multi_agent.tools)} tool sources')


## Cell 7: Run Agent with ADK Runner


In [ ]:
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

# Note: This requires the MCP server running locally
# Start it first: python documind_server.py

async def test_agent():
    session_service = InMemorySessionService()
    runner = Runner(
        agent=agent,  # or multi_agent
        app_name='documind',
        session_service=session_service,
    )
    session = await session_service.create_session(
        app_name='documind', user_id='student')

    # Send a query
    content = types.Content(
        role='user',
        parts=[types.Part.from_text(text='What documents do we have?')])
    
    print('Sending query to agent...')
    async for event in runner.run_async(
        user_id='student', session_id=session.id, new_message=content):
        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.text:
                    print(f'Agent: {part.text}')
                if part.function_call:
                    print(f'Tool call: {part.function_call.name}')

# Run (requires server running)
# await test_agent()


## ✅ Lesson 7.3 Complete! MODULE 7 COMPLETE!

**Agent ↔ MCP connection mastered:**
- ✅ McpToolset with StreamableHTTPConnectionParams
- ✅ IAM authentication with OIDC identity tokens
- ✅ Cloud Run Proxy for local development
- ✅ BigQuery MCP via Toolbox + BQML predictions
- ✅ Custom Document AI FastMCP server
- ✅ Multi-server agent with tool_filter
- ✅ Gemini routes across all tool sources automatically

**Module 7 Complete — 3 Lessons:**
- 7.1: Built FastMCP server with 4 tools + Streamable HTTP
- 7.2: Deployed to Cloud Run with IAM + scale-to-zero + Toolbox
- 7.3: Connected ADK agents to remote MCP servers + BQ + Doc AI

**Next: Module 8 — AI Agents with ADK, Agent Engine, A2A**
